# Aube T1: full-flood rollout

Evaluation of the regular 40 m and multimesh 40-80-160 m source-node models. The checkpoint is selected from the minimum saved training loss.

In [ ]:
from pathlib import Path
import math
import pickle
import re
import sys

import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
import torch
from matplotlib.colors import ListedColormap
from omegaconf import OmegaConf


def find_project_root():
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (path / "python").exists() and (path / "bin").exists():
            return path
    raise RuntimeError("Project root not found.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from modulus.launch.utils import load_checkpoint
from python.CustomMeshGraphNet import MeshGraphNetWithSourceNodes
from python.create_dgl_dataset import TelemacDatasetWithSourceNodes
from python.python_code.data_manip.extraction.telemac_file import TelemacFile

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# User parameters
CONFIG_DIR = PROJECT_ROOT / "bin/conf/research"

# Replace these two paths with the SLURM .out or train .log files.
METHOD_SPECS = [
    {
        "name": "Regular 40 m",
        "config": CONFIG_DIR / "config_overfit_Aube_T1_ghost_40m_useQ.yaml",
        "loss_log": Path("/path/to/regular_40m_training.log"),
        "color": "#1f77b4",
    },
    {
        "name": "Multimesh 40-80-160 m",
        "config": CONFIG_DIR / "config_overfit_Aube_T1_ghost_40m_useQ_multi.yaml",
        "loss_log": Path("/path/to/multimesh_40_160_training.log"),
        "color": "#d95f02",
    },
]

EVENTS = [
    {
        "name": "Q5 overfit",
        "regular_dynamic": "/work/m24046/m24046mrcr/Test_Aube_T1_regular/dataset_40m_ghost/regular_short/08_T1_V9_Topo3_KV5_Q5_0_47-76_interpolated.pkl",
        "hydro": "/work/m24046/m24046mrcr/Test_Aube_T1_regular/original_dataset/T1_Q5_V1.liq",
    },
]

PLOT_METHOD_NAMES = ["Regular 40 m", "Multimesh 40-80-160 m"]
REGULAR_MESH_PATH = "/work/m24046/m24046mrcr/Test_Aube_T1_regular/dataset_40m/Aube_regular_T1_40m.slf"

DT_SECONDS = 7200.0
REQUESTED_MAX_HOURS = 58
THRESHOLD_M = 0.05
CHECKPOINT_EVERY = 10

QUAL_EVENT_NAME = "Q5 overfit"
QUAL_HOURS = [12, 24, 48]

In [ ]:
LOSS_PATTERN = re.compile(r"epoch:\s*(\d+).*?loss:\s*([0-9.eE+-]+)")


def best_saved_checkpoint(log_path):
    matches = LOSS_PATTERN.findall(Path(log_path).read_text(errors="replace"))
    saved_losses = [
        (int(epoch), float(loss))
        for epoch, loss in matches
        if int(epoch) % CHECKPOINT_EVERY == 0
    ]
    return min(saved_losses, key=lambda item: item[1])


def load_method(spec):
    epoch, loss = best_saved_checkpoint(spec["loss_log"])
    method = dict(spec)
    method["cfg"] = OmegaConf.load(spec["config"])
    method["epoch"] = epoch
    method["loss"] = loss
    return method


METHODS = [load_method(spec) for spec in METHOD_SPECS]
METHOD_BY_NAME = {method["name"]: method for method in METHODS}

for method in METHODS:
    print(f'{method["name"]}: epoch={method["epoch"]}, training loss={method["loss"]:.6e}')

In [ ]:
def dynamic_length(path):
    with open(path, "rb") as handle:
        return len(pickle.load(handle))


available_steps = min(dynamic_length(event["regular_dynamic"]) for event in EVENTS) - 1
requested_steps = int(REQUESTED_MAX_HOURS * 3600.0 // DT_SECONDS)
MAX_STEPS = min(available_steps, requested_steps)
HORIZONS_STEPS = list(range(1, MAX_STEPS + 1))
HOURS = np.asarray(HORIZONS_STEPS, dtype=float) * DT_SECONDS / 3600.0
SEQUENCE_LENGTH = MAX_STEPS + 1

print(f"events={len(EVENTS)} | steps=1..{MAX_STEPS} | hours={HOURS[0]:g}..{HOURS[-1]:g}")

In [ ]:
def build_dataset(method, event, sequence_length):
    cfg = method["cfg"]
    return TelemacDatasetWithSourceNodes(
        name=f'eval_{event["name"]}',
        data_dir=str(cfg.data_dir),
        dynamic_data_files=[event["regular_dynamic"]],
        hydro_data_files=[event["hydro"]],
        inlet_node_lists=OmegaConf.to_container(cfg.inlet_node_lists),
        use_q_feature=bool(cfg.use_q_feature),
        split="test",
        ckpt_path=str(cfg.ckpt_path),
        normalize=True,
        sequence_length=sequence_length,
        overlap=0,
        dt_seconds=float(cfg.dt_seconds),
    )


def build_model(method, dataset):
    cfg = method["cfg"]
    model = MeshGraphNetWithSourceNodes(
        input_dim_nodes_phys=dataset.base_graph.ndata["static"].shape[1] + dataset.physical_dynamic_dim,
        input_dim_nodes_src=len(dataset.source_feature_names),
        input_dim_edges=int(cfg.num_edge_features),
        output_dim=int(cfg.num_output_features),
        processor_size=int(cfg.mp_layers),
        hidden_dim_processor=64,
        hidden_dim_node_encoder=64,
        hidden_dim_edge_encoder=64,
        hidden_dim_node_decoder=64,
        do_concat_trick=bool(cfg.do_concat_trick),
        num_processor_checkpoint_segments=int(cfg.num_processor_checkpoint_segments),
    )
    load_checkpoint(str(cfg.ckpt_path), models=model, device=device, epoch=method["epoch"])
    model.to(device).eval()
    return model


def build_context(dataset):
    stats = dataset.node_stats
    return {
        "static_dim": dataset.base_graph.ndata["static"].shape[1],
        "physical_dynamic_dim": dataset.physical_dynamic_dim,
        "state_mean": torch.tensor([stats["h"].item(), stats["u"].item(), stats["v"].item()], device=device),
        "state_std": torch.tensor([stats["h_std"].item(), stats["u_std"].item(), stats["v_std"].item()], device=device),
        "delta_mean": torch.tensor([stats["delta_h"].item(), stats["delta_u"].item(), stats["delta_v"].item()], device=device),
        "delta_std": torch.tensor([stats["delta_h_std"].item(), stats["delta_u_std"].item(), stats["delta_v_std"].item()], device=device),
    }


def rollout_sequence(model, dataset, sequence_index=0):
    sequence = dataset[sequence_index]
    context = build_context(dataset)
    state = {
        "graph": sequence[0]["graph"].to(device),
        "x_phys": sequence[0]["x_phys"].to(device),
        "x_src": sequence[0]["x_src"].to(device),
    }
    predictions = []
    targets = []

    for next_step in sequence[1:]:
        with torch.no_grad():
            y_pred_n = model(
                graph=state["graph"],
                x_phys=state["x_phys"],
                x_src=state["x_src"],
                edge_features=state["graph"].edata["x"].to(device),
            )

        start = context["static_dim"]
        x_t = state["x_phys"][:, start:start + 3] * context["state_std"] + context["state_mean"]
        y_pred = y_pred_n * context["delta_std"] + context["delta_mean"]
        x_pred = x_t + y_pred
        x_gt_n = next_step["x_phys"][:, start:start + 3].to(device)
        x_gt = x_gt_n * context["state_std"] + context["state_mean"]

        h_mask = (state["x_phys"][:, :4] == torch.tensor([0, 1, 0, 0], device=device)).all(dim=1)
        x_pred = x_pred.clone()
        x_pred[h_mask, 0] = x_gt[h_mask, 0]
        predictions.append(x_pred.cpu().numpy())
        targets.append(x_gt.cpu().numpy())

        x_next_n = (x_pred - context["state_mean"]) / (context["state_std"] + 1e-12)
        static = state["x_phys"][:, :start]
        forcing = next_step["x_phys"][:, start + 3:start + context["physical_dynamic_dim"]].to(device)
        state = {
            "graph": next_step["graph"].to(device),
            "x_phys": torch.cat((static, x_next_n, forcing), dim=1),
            "x_src": next_step["x_src"].to(device),
        }

    return np.stack(predictions), np.stack(targets)


def csi(prediction, target, threshold):
    pred_wet = prediction >= threshold
    target_wet = target >= threshold
    tp = np.logical_and(pred_wet, target_wet).sum()
    fp = np.logical_and(pred_wet, ~target_wet).sum()
    fn = np.logical_and(~pred_wet, target_wet).sum()
    return float(tp / (tp + fp + fn)) if tp + fp + fn else math.nan

In [ ]:
def evaluate_method(method):
    first_dataset = build_dataset(method, EVENTS[0], SEQUENCE_LENGTH)
    model = build_model(method, first_dataset)
    event_payloads = {}

    for event in EVENTS:
        dataset = build_dataset(method, event, SEQUENCE_LENGTH)
        prediction, target = rollout_sequence(model, dataset)
        event_payloads[event["name"]] = {
            "prediction": prediction,
            "target": target,
            "l1": np.mean(np.abs(prediction - target), axis=1),
            "csi": np.asarray([
                csi(prediction[step, :, 0], target[step, :, 0], THRESHOLD_M)
                for step in range(MAX_STEPS)
            ]),
        }

    l1 = np.stack([payload["l1"] for payload in event_payloads.values()])
    csi_values = np.stack([payload["csi"] for payload in event_payloads.values()])
    return {
        "events": event_payloads,
        "l1_mean": np.nanmean(l1, axis=0),
        "l1_std": np.nanstd(l1, axis=0),
        "csi_mean": np.nanmean(csi_values, axis=0),
        "csi_std": np.nanstd(csi_values, axis=0),
    }


RESULTS = {method["name"]: evaluate_method(method) for method in METHODS}

In [ ]:
def plot_band(ax, mean, std, method, label):
    ax.plot(HOURS, mean, color=method["color"], linewidth=2, label=label)
    ax.fill_between(HOURS, mean - std, mean + std, color=method["color"], alpha=0.18)


fig, ax = plt.subplots(figsize=(10, 4.5))
for name in PLOT_METHOD_NAMES:
    plot_band(ax, RESULTS[name]["csi_mean"], RESULTS[name]["csi_std"], METHOD_BY_NAME[name], name)
ax.set(title=f"Full-flood CSI at {THRESHOLD_M:.2f} m", xlabel="Horizon (hours)", ylabel="CSI", ylim=(0, 1))
ax.grid(alpha=0.3)
ax.legend(frameon=False)
plt.show()

component_labels = ["L1 h (m)", "L1 u (m/s)", "L1 v (m/s)"]
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
for component, ax in enumerate(axes):
    for name in PLOT_METHOD_NAMES:
        plot_band(
            ax,
            RESULTS[name]["l1_mean"][:, component],
            RESULTS[name]["l1_std"][:, component],
            METHOD_BY_NAME[name],
            name,
        )
    ax.set_ylabel(component_labels[component])
    ax.grid(alpha=0.3)
axes[0].legend(frameon=False)
axes[-1].set_xlabel("Horizon (hours)")
fig.suptitle("Full-flood L1 mean +/- std")
fig.tight_layout()
plt.show()

In [ ]:
mesh = TelemacFile(REGULAR_MESH_PATH)
triangles = np.asarray(mesh.ikle2, dtype=np.int64)
triangles = triangles - triangles.min()
tri = mtri.Triangulation(np.asarray(mesh.meshx), np.asarray(mesh.meshy), triangles)
cmap = ListedColormap(["white", "#1f4e79"])


def node_to_face(values):
    return values[tri.triangles].max(axis=1).astype(float)


qual_steps = [int(hour * 3600.0 / DT_SECONDS) for hour in QUAL_HOURS]
fig, axes = plt.subplots(
    1 + len(PLOT_METHOD_NAMES),
    len(qual_steps),
    figsize=(4.3 * len(qual_steps), 3.6 * (1 + len(PLOT_METHOD_NAMES))),
    squeeze=False,
)

reference = RESULTS[PLOT_METHOD_NAMES[0]]["events"][QUAL_EVENT_NAME]["target"]
for column, step in enumerate(qual_steps):
    axes[0, column].tripcolor(
        tri,
        facecolors=node_to_face(reference[step - 1, :, 0] >= THRESHOLD_M),
        shading="flat",
        cmap=cmap,
        vmin=0,
        vmax=1,
    )
    axes[0, column].set_title(f"Reference | {step * DT_SECONDS / 3600:g} h")

    for row, name in enumerate(PLOT_METHOD_NAMES, start=1):
        payload = RESULTS[name]["events"][QUAL_EVENT_NAME]
        axes[row, column].tripcolor(
            tri,
            facecolors=node_to_face(payload["prediction"][step - 1, :, 0] >= THRESHOLD_M),
            shading="flat",
            cmap=cmap,
            vmin=0,
            vmax=1,
        )
        axes[row, column].set_title(f'{name} | CSI={payload["csi"][step - 1]:.3f}')

for ax in axes.flat:
    ax.set_aspect("equal")
    ax.set_axis_off()
fig.suptitle(f"{QUAL_EVENT_NAME} | threshold={THRESHOLD_M:.2f} m")
fig.tight_layout()
plt.show()